In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "auto")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

In [0]:
%pip install -q xgboost scikit-learn pandas numpy pyarrow mosaicml-streaming
dbutils.library.restartPython()

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
import numpy as np
import os
import pandas as pd
import xgboost as xgb
from sklearn.metrics import ndcg_score, roc_auc_score
from streaming import StreamingDataset
import mlflow
from tqdm import tqdm
import shap
import matplotlib.pyplot as plt
from pyspark.sql import DataFrame
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
import optuna
import joblib
import config

In [0]:
%sql
create or replace table ds_sandbox.dev_adrienne_pred_ranked as
with t0 as (
  select *,
  case when theme_clean like '%women%' 
      and GREATEST(Womenswearconfidence_score,Menswearconfidence_score,Beautyconfidence_score,Homeconfidence_score) == Womenswearconfidence_score
      then 1
  when theme_clean like '%girls%|%boys%'
      and GREATEST(Womenswearconfidence_score,Menswearconfidence_score,Beautyconfidence_score,Homeconfidence_score,Familyconfidence_score) == Familyconfidence_score
      THEN 1
  when theme_clean like '%men%'
      and GREATEST(Womenswearconfidence_score,Menswearconfidence_score,Beautyconfidence_score,Homeconfidence_score) == Menswearconfidence_score
      THEN 1
  when theme_clean like '%beauty%'
      and GREATEST(Womenswearconfidence_score,Menswearconfidence_score,Beautyconfidence_score,Homeconfidence_score) == Beautyconfidence_score
      THEN 1
  when theme_clean like '%home%'
      and GREATEST(Womenswearconfidence_score,Menswearconfidence_score,Beautyconfidence_score,Homeconfidence_score) == Homeconfidence_score
      THEN 1
  else 0
  end as same_dept
  from ds_sandbox.dev_adrienne_pred_complete  
  -- where account_number = 'M52K2576'
  )

  , t1 as(
  select *,
  row_number() OVER (
      PARTITION BY account_number, reference_date
      ORDER BY 
      -- Primary: Recent behavior wins
      CASE WHEN views_behavior__recency = 0 THEN 999999 ELSE views_behavior__recency END ASC,
      CASE WHEN atbs_behavior__recency = 0 THEN 999999 ELSE atbs_behavior__recency END ASC,
      repurchase_ratio DESC,      
      -- Secondary: Frequency and consensus
      num_retrieval_methods DESC,
      baskets_behavior__frequency DESC,
      atbs_behavior__frequency DESC,
      views_behavior__frequency DESC,
      
      -- Tertiary: Association strength
      COALESCE(algo_baskets5__lift_top10, 0) DESC,
      COALESCE(algo_atbs5__lift_top10, 0) DESC,
      COALESCE(algo_views5__lift_top10, 0) DESC,

      -- dept segment
      same_dept desc,

      theme_clean  -- Deterministic tiebreaker
  ) AS simple_rules_rank
  from t0
  )
  , final as (
  select *,
  case
  when views_behavior__recency >0 and views_behavior__recency <9999 then 'views_recency'
  when atbs_behavior__recency >0 and atbs_behavior__recency <9999 then 'atbs_recency'
  when repurchase_ratio >0 then 'repurchase_ratio'
  when num_retrieval_methods >0 and baskets_behavior__frequency >0 then 'ret_met_baskets_frequency'
  when num_retrieval_methods >0 and atbs_behavior__frequency >0 then 'ret_met_atbs_frequency'
  when num_retrieval_methods >0 and views_behavior__frequency >0 then 'ret_met_views_frequency'
  when num_retrieval_methods >0 and algo_baskets5__lift_top10 >0 then 'ret_met_algo_baskets5_lift_top10'
  when num_retrieval_methods >0 and algo_atbs5__lift_top10 >0 then 'ret_met_algo_atbs5_lift_top10'
  when num_retrieval_methods >0 and algo_views5__lift_top10 >0 then 'ret_met_algo_views5_lift_top10'
  when num_retrieval_methods >0 and same_dept = 1 then 'ret_met_same_dept'
  when baskets_behavior__frequency >0 then 'baskets_frequency'
  when atbs_behavior__frequency >0 then 'atbs_frequency'
  when views_behavior__frequency >0 then 'views_frequency'
  when algo_baskets5__lift_top10 >0 then 'algo_baskets5_lift_top10'
  when algo_atbs5__lift_top10 >0 then 'algo_atbs5_lift_top10'
  when algo_views5__lift_top10 >0 then 'algo_views5_lift_top10'
  when same_dept = 1 then 'same_dept'
  else 'theme'
  end as rules_rank_source
  from t1
  -- where account_number = 'M52K2576'
  GROUP BY all
  ORDER BY simple_rules_rank
  )

  select distinct 
  *
  from final
  -- where simple_rules_rank <= 100
  order by account_number, simple_rules_rank

In [0]:
base = spark.sql('select * from ds_sandbox.dev_adrienne_pred_ranked where simple_rules_rank <=25')

base.cache().count()

In [0]:
# split data into smaller batches on account_number
account_splits = (
    base.select("account_number")
    .distinct()
    .withColumn("split_id", (F.row_number().over(Window.orderBy("account_number")) % 10))
)

base_with_split = base.join(account_splits, on="account_number", how="left")

dfs = [base_with_split.filter(F.col("split_id") == i).drop("split_id") for i in range(10)]

# call in feature cols and select only feature cols
features = config.features
# features = ['month', 'algo_atbs5__cs_top10', 'algo_baskets5__cs_top10', 'algo_baskets5__freq12_norm_top10', 'algo_views1__freq12_norm_top10', 'algo_views5__freq12_norm_top10', 'views_behavior__recency', 'views_behavior__most_recent', 'views_behavior__recency_rank', 'baskets_behavior__frequency', 'repurchase_stage', 'user_theme_breadth', 'views_ly_7', 'baskets_ly_7', 'baskets_ly_30', 'trending_7x30', 'simple_rules_rank']

df1 = dfs[0]
df2 = dfs[1]
df3 = dfs[2]
df4 = dfs[3]
df5 = dfs[4]
df6 = dfs[5]
df7 = dfs[6]
df8 = dfs[7]
df9 = dfs[8]
df10 = dfs[9]

In [0]:
dfs_list = [df1, df2, df3, df4, df5, df6, df7, df8, df9, df10]
X_dfs = [
    df.select('account_number', 'theme_clean', *features).toPandas()
    for df in dfs_list
]
X_df1, X_df2, X_df3, X_df4, X_df5, X_df6, X_df7, X_df8, X_df9, X_df10 = X_dfs

In [0]:
def prepare_xgb_data(df,feature_cols, encoders):
    """
    Prepare data for XGBoost LTR.
    
    Returns:
        X: Feature matrix
        y: Labels
        groups: Group sizes for ranking (items per user)
        feature_cols: List of feature column names
    """
    # Make a copy to avoid modifying original
    df = df.copy()
    
    if encoders is None:
        encoders = {}
        is_training = True
    else:
        is_training = False

    for col in feature_cols:
        if df[col].dtype == 'object' or df[col].dtype.name == 'category':
            if is_training:
                # TRAINING: Create and fit new encoder
                le = LabelEncoder()
                df[col] = le.fit_transform(df[col].astype(str))
                encoders[col] = le
            else:
                # VAL/TEST: Use existing encoder (THE VECTORIZED FIX)
                le = encoders[col]
                
                # 1. Check which values exist in the training "alphabet"
                valid_mask = df[col].astype(str).isin(le.classes_)
                
                # 2. Create a temporary series where "New" values are safely replaced by a known label
                # (We use le.classes_[0] just as a placeholder to prevent the .transform() crash)
                safe_series = np.where(valid_mask, df[col].astype(str), le.classes_[0])
                
                # 3. Transform the whole column at once (Lightning Fast)
                df[col] = le.transform(safe_series)
                
                # 4. Set the previously "New" values to -1 so the model knows they are unknown
                df.loc[~valid_mask, col] = -1
                print('encoded')
    
    # Sort by account_number to ensure items are grouped together
    df_sorted = df.sort_values('account_number').reset_index(drop=True)
    
    # Extract features and labels
    X = df_sorted[feature_cols].values.astype(np.float32)
    
    # Compute group sizes (number of items per user)
    groups = df_sorted.groupby('account_number').size().values
    
    print(f"Features shape: {X.shape}")
    print(f"Groups: {len(groups)} users, {np.mean(groups):.1f} items/user avg")
    
    return X, groups, feature_cols

In [0]:
encoders = joblib.load("ranking_encoders.joblib")

X_train1, groups_train1, feature_cols = prepare_xgb_data(X_df1,features,encoders)
X_train2, groups_train2, feature_cols = prepare_xgb_data(X_df2,features,encoders)
X_train3, groups_train3, feature_cols = prepare_xgb_data(X_df3,features,encoders)
X_train4, groups_train4, feature_cols = prepare_xgb_data(X_df4,features,encoders)
X_train5, groups_train5, feature_cols = prepare_xgb_data(X_df5,features,encoders)
X_train6, groups_train6, feature_cols = prepare_xgb_data(X_df6,features,encoders)
X_train7, groups_train7, feature_cols = prepare_xgb_data(X_df7,features,encoders)
X_train8, groups_train8, feature_cols = prepare_xgb_data(X_df8,features,encoders)
X_train9, groups_train9, feature_cols = prepare_xgb_data(X_df9,features,encoders)
X_train10, groups_train10, feature_cols = prepare_xgb_data(X_df10,features,encoders)


In [0]:
dpredict1 = xgb.QuantileDMatrix(X_train1, feature_names=feature_cols)
dpredict2 = xgb.QuantileDMatrix(X_train2, feature_names=feature_cols)
dpredict3 = xgb.QuantileDMatrix(X_train3, feature_names=feature_cols)
dpredict4 = xgb.QuantileDMatrix(X_train4, feature_names=feature_cols)
dpredict5 = xgb.QuantileDMatrix(X_train5, feature_names=feature_cols)
dpredict6 = xgb.QuantileDMatrix(X_train6, feature_names=feature_cols)
dpredict7 = xgb.QuantileDMatrix(X_train7, feature_names=feature_cols)
dpredict8 = xgb.QuantileDMatrix(X_train8, feature_names=feature_cols)
dpredict9 = xgb.QuantileDMatrix(X_train9, feature_names=feature_cols)
dpredict10 = xgb.QuantileDMatrix(X_train10, feature_names=feature_cols)

In [0]:
# transform and predict 
# import mlflow.xgboost

# Replace with the Run ID you found in the UI
run_id = "9d97fe1f03eb4052bf660bda30fe6000"
model_uri = f"runs:/{run_id}/model"

# This loads model2 back into memory exactly as it was
model = mlflow.xgboost.load_model(model_uri)

# Now you can predict immediately
preds1 = model.predict(dpredict1)
preds2 = model.predict(dpredict2)
preds3 = model.predict(dpredict3)
preds4 = model.predict(dpredict4)
preds5 = model.predict(dpredict5)
preds6 = model.predict(dpredict6)
preds7 = model.predict(dpredict7)
preds8 = model.predict(dpredict8)
preds9 = model.predict(dpredict9)
preds10 = model.predict(dpredict10)

In [0]:
def get_sorted_predictions(X_df, preds, theme_col='theme_clean'):
    """
    Returns a DataFrame with account_number, theme, month, and prediction,
    sorted by account_number and descending prediction.
    """
    df_sorted = X_df.sort_values('account_number').reset_index(drop=True)
    results_df = pd.DataFrame({
        'account_number': df_sorted['account_number'],
        'theme': df_sorted[theme_col],
        'month': df_sorted['month'],
        'prediction': preds
    })
    results_df = results_df.sort_values(['account_number', 'prediction'], ascending=[True, False])
    # display(results_df.head(20))
    return results_df

results_df1 = get_sorted_predictions(X_df1, preds1)
results_df2 = get_sorted_predictions(X_df2, preds2)
results_df3 = get_sorted_predictions(X_df3, preds3)
results_df4 = get_sorted_predictions(X_df4, preds4)
results_df5 = get_sorted_predictions(X_df5, preds5)
results_df6 = get_sorted_predictions(X_df6, preds6)
results_df7 = get_sorted_predictions(X_df7, preds7)
results_df8 = get_sorted_predictions(X_df8, preds8)
results_df9 = get_sorted_predictions(X_df9, preds9)
results_df10 = get_sorted_predictions(X_df10, preds10)

In [0]:
# join all preds back together
full_results = pd.concat([results_df1, results_df2, results_df3, results_df4, results_df5, results_df6, results_df7, results_df8, results_df9, results_df10], ignore_index=True)
# display(full_results)

In [0]:
full_results = full_results.rename(columns={'prediction': 'ProbAggRebased',
                                              'account_number': 'AccountNumber',
                                              'theme': 'NextTheme'})
full_results['rundate'] = pd.Timestamp.now().date()
display(full_results)

In [0]:
# limit to top 50 ads and account_number, theme, ..{relevant cols}
# clean up themeclean > theme
theme_mapping = spark.sql("SELECT distinct theme, regexp_replace(theme, '[^a-zA-Z0-9]', '') as theme_clean FROM warehouse.next_uk_nextads_item_themes_latest where theme_rank = 1")

full_results_spark = spark.createDataFrame(full_results)
full_df_fixed = (full_results_spark.join(theme_mapping, full_results_spark['NextTheme'] == theme_mapping['theme_clean'], how='left')
                 .select('AccountNumber','theme','ProbAggRebased','rundate'))
fixed = full_df_fixed.withColumnRenamed('theme','NextTheme')
display(fixed)

In [0]:
# validation
# check acc has more than X themes
acc_to_theme = fixed.groupBy('AccountNumber').agg(F.count_distinct('NextTheme').alias('theme_count')).filter(F.col('theme_count') < 25)
if acc_to_theme.count() > 0:
    print(f'WARN: {acc_to_theme.count()} accounts have less than 25 themes assigned')
    display(acc_to_theme)

# check there are no dupe acc:theme pairs
acc_theme_pairs = fixed.groupBy('AccountNumber', 'NextTheme').agg(F.count('*').alias('count'))
if acc_theme_pairs.filter(F.col('count') > 1).count() > 0:
    print('WARN: there are duplicate account_number:theme pairs')
    display(acc_theme_pairs.filter(F.col('count') > 1).sort(F.desc('count')))

# check all acc nums from base are present
all_accounts = (
    spark.read.table('warehouse.baskets_uk_3y')
    .filter(F.col('order_date') > F.date_sub(F.current_date(), 365))
    .select('account_number')
    .distinct()
)
if all_accounts.join(fixed.select('AccountNumber').distinct(), all_accounts['account_number'] == F.col('AccountNumber'), how='leftanti').count() > 0:
    print('ERROR: not all accounts from base are present in the model output')
    display(all_accounts.join(fixed.select('AccountNumber').distinct(), all_accounts['account_number'] == F.col('AccountNumber'), how='leftanti'))

# pull in yesterdays top 20 assigned themes, if difference > 20 message, if > 50 error
# top_themes = (
#     full_results.sort_values(['AccountNumber', 'ProbAggRebased'], ascending=[True, False])
#     .groupby('AccountNumber')
#     .first()
#     .reset_index()[['AccountNumber', 'NextTheme', 'ProbAggRebased']]
# )
# display(top_themes.groupby('NextTheme').size().reset_index(name='count').sort_values(by='count', ascending=False))

# top_themes_yest = (
#     full_results.sort_values(['AccountNumber', 'ProbAggRebased'], ascending=[True, False])
#     .groupby('AccountNumber')
#     .first()
#     .reset_index()[['AccountNumber', 'NextTheme', 'ProbAggRebased']]
# )
# display(top_themes_yest.groupby('NextTheme').size().reset_index(name='count').sort_values(by='count', ascending=False))


In [0]:
fixed = fixed[['AccountNumber', 'NextTheme', 'ProbAggRebased', 'rundate']]
fixed.write.mode("overwrite").saveAsTable("ds_sandbox.next_uk_next_ads_hackathon_model_latest")
fixed.write.mode("append").saveAsTable("ds_sandbox.next_uk_next_ads_hackathon_model")